# EEEM004 MSc Project — Spectrogram Segmentation Pipeline (SAM)
**Automated Soundtrack Personalisation for Neurodivergent Listeners Using Generative AI**

---
**Student:** Louis Ilett  
**Supervisor:** Prof Philip Jackson

---
## Pipeline Overview
1. Audio acquisition (YouTube)
2. Convert audio → spectrogram
3. Apply Meta SAM for segmentation
4. Mask selected regions
5. Reconstruct modified audio


## 1. Setup

In [ ]:
!pip install librosa matplotlib soundfile yt-dlp opencv-python --quiet
!pip install git+https://github.com/facebookresearch/segment-anything.git --quiet

print("Install complete — upload SAM checkpoint next")

## 2. Imports

In [ ]:
import os
import cv2
import torch
import numpy as np
import librosa
import librosa.display
import soundfile as sf
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from segment_anything import sam_model_registry, SamPredictor

print("Imports successful")

## 3. Upload SAM Checkpoint
Download from:
https://github.com/facebookresearch/segment-anything
Upload `sam_vit_b.pth` to Colab

In [ ]:
CHECKPOINT_PATH = "/content/sam_vit_b.pth"  # update if needed
assert os.path.exists(CHECKPOINT_PATH), "Upload SAM checkpoint first"

## 4. Audio Acquisition

In [ ]:
YOUTUBE_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"
INPUT_PATH = "/content/audio.mp3"

!yt-dlp -x --audio-format mp3 -o "{INPUT_PATH}" "{YOUTUBE_URL}"

display(Audio(INPUT_PATH))

## 5. Convert to Spectrogram

In [ ]:
y, sr = librosa.load(INPUT_PATH, sr=None)

S = librosa.stft(y)
S_mag = np.abs(S)
S_db = librosa.amplitude_to_db(S_mag, ref=np.max)

plt.figure(figsize=(12,5))
librosa.display.specshow(S_db, sr=sr, x_axis='time', y_axis='log')
plt.title("Spectrogram")
plt.colorbar()
plt.show()

## 6. Convert Spectrogram to Image

In [ ]:
spec_norm = (S_db - S_db.min()) / (S_db.max() - S_db.min())
spec_img = (spec_norm * 255).astype(np.uint8)
spec_rgb = cv2.cvtColor(spec_img, cv2.COLOR_GRAY2RGB)

cv2.imwrite("/content/spec.png", spec_rgb)

plt.imshow(spec_rgb)
plt.title("Spectrogram Image")
plt.axis('off')
plt.show()

## 7. Load SAM

In [ ]:
sam = sam_model_registry["vit_b"](checkpoint=CHECKPOINT_PATH)
predictor = SamPredictor(sam)
predictor.set_image(spec_rgb)

print("SAM loaded")

## 8. Segment Region (Manual Prompt)
Adjust the coordinates to target different sounds

In [ ]:
input_point = np.array([[500, 200]])
input_label = np.array([1])

masks, scores, _ = predictor.predict(
    point_coords=input_point,
    point_labels=input_label,
    multimask_output=True
)

mask = masks[0]

plt.imshow(mask, cmap='gray')
plt.title("SAM Mask")
plt.axis('off')
plt.show()

## 9. Apply Mask + Reconstruct Audio

In [ ]:
mask_resized = cv2.resize(mask.astype(np.float32), (S_mag.shape[1], S_mag.shape[0]))

masked_mag = S_mag * mask_resized

reconstructed = librosa.istft(masked_mag * np.exp(1j * np.angle(S)))

OUTPUT_PATH = "/content/reconstructed.wav"
sf.write(OUTPUT_PATH, reconstructed, sr)

print("Reconstruction complete")

display(Audio(OUTPUT_PATH))

## 10. Observations
Document what regions were selected and how the audio changed.